In [2]:
# 학번과 이름을 출력하세요
# 예시
# 학번 = '30261240'
# 이름 = '이현정'

학번 = '20242028'
이름 = '유건'

print(20242028,'유건')

20242028 유건


In [4]:
from abc import ABC, abstractmethod
from collections import namedtuple

Customer = namedtuple('Customer', 'name fidelity')
park = Customer('Park',100)
park

Customer(name='Park', fidelity=100)

In [5]:
class LineItem:
  def __init__(self,product,quantity,price):
    self.product = product
    self.quantity = quantity
    self.price = price

  def total(self):
    return self.price * self.quantity


In [15]:
class Order:  # Context
    def __init__(self, customer, cart, promotion=None):
        self.customer = customer
        self.cart = list(cart)
        self.promotion = promotion  # 할인 객체

    def total(self):
        if not hasattr(self, '__total'):
            self.__total = sum(item.total() for item in self.cart)
        return self.__total

    def due(self):
        if self.promotion is None:
            discount = 0
        else:
            discount = self.promotion.discount(self)  # self == Order 객체
        return self.total() - discount

    def __repr__(self):
        fmt = '<Order total: {:.2f} due: {:.2f}>'
        return fmt.format(self.total(), self.due())


In [16]:
class Promotion(ABC):
  @abstractmethod
  def discount(self,order):
    pass

In [17]:
class FidelityPromo(Promotion):

    def discount(self, order):
        return order.total() * 0.05 if order.customer.fidelity >= 1000 else 0

class BulkItemPromo(Promotion):

    def discount(self, order):
        discount = 0
        for item in order.cart:
            if item.quantity >= 20:
                discount += item.total() * 0.1
        return discount

class LargeOrderPromo(Promotion):

    def discount(self, order):
        distinct_items = {item.product for item in order.cart}
        if len(distinct_items) >= 10:
            return order.total() * 0.07
        return 0

In [18]:
joe = Customer('John Doe',0)
ann = Customer('Ann Smith',1100)
cart = [
  LineItem('banana',4, .5),
  LineItem('apple',10,1.5),
  LineItem('watermellon',5,5.0)
]

In [19]:
Order(joe,cart,FidelityPromo())

<Order total: 42.00 due: 42.00>

In [21]:
Order(ann,cart,FidelityPromo())

<Order total: 42.00 due: 39.90>

In [22]:
banana_cart = [LineItem('banana',30, .5),
               LineItem('apple',10,1.5)]

In [23]:
Order(joe, banana_cart, BulkItemPromo())

<Order total: 30.00 due: 28.50>

In [24]:
Long_order = [LineItem(str(item_code),1,1.0) for item_code in range(10)]

In [25]:
Order(joe,Long_order,LargeOrderPromo())

<Order total: 10.00 due: 9.30>

In [27]:
Order(joe,cart,LargeOrderPromo())

<Order total: 42.00 due: 42.00>

In [28]:
class Order:
  def __init__(self,customer,cart,promotion=None):
    self.customer = customer
    self.cart = list(cart)
    self.promotion = promotion

  def total(self):
    if not hasattr(self,'__total'):
      self.__total = sum(item.total() for item in self.cart)
    return self.__total

  def due(self):
    if self.promotion is None:
      discount = 0
    else:
      discount = self.promotion(self)
    return self.total() - discount

  def __repr__(self):
    fmt = '<Order total: {:.2f} due: {:.2f}>'
    return fmt.format(self.total(),self.due())

In [29]:
def fidelity_promo(order):
  return order.total() * 0.05 if order.customer.fidelity >= 1000 else 0

def bulk_item_promo(order):
  discount = 0
  for item in order.cart:
    if item.quantity >= 20:
      discount += item.total() * 0.1
  return discount

def large_order_promo(order):
  distinct_items = {item.product for item in order.cart}
  if len(distinct_items) >= 10:
    return order.total() * 0.07
  return 0

In [30]:
Order(joe,cart,fidelity_promo)

<Order total: 42.00 due: 42.00>

In [31]:
Order(ann,cart, fidelity_promo)

<Order total: 42.00 due: 39.90>

In [32]:
Order(joe,banana_cart,bulk_item_promo)

<Order total: 30.00 due: 28.50>

In [33]:
Order(ann, Long_order, large_order_promo)

<Order total: 10.00 due: 9.30>

In [40]:
promos = [fidelity_promo,bulk_item_promo,large_order_promo]

def best_promo(order):
  return max(promo(order) for promo in promos)

In [41]:
Order(joe,Long_order, best_promo)

<Order total: 10.00 due: 9.30>

In [43]:
Order(joe,banana_cart, best_promo)

<Order total: 30.00 due: 28.50>

In [42]:
Order(ann,cart, best_promo)

<Order total: 42.00 due: 39.90>

In [45]:
promos = [globals()[name] for name in globals()
          if name.endswith('_promo')
          and name != 'best_promo']
promos

[<function __main__.fidelity_promo(order)>,
 <function __main__.bulk_item_promo(order)>,
 <function __main__.large_order_promo(order)>]